# Seeing Things: UFO Reports as Human Reporting Data

This notebook is the technical explainer behind the public website. The website tells the story for a non-technical audience; this notebook documents the data source, cleaning choices, analysis logic, narrative genre, limitations, and references.

## 1. Motivation

The project treats UFO sightings as social data. A UFO report is not only a claim about something in the sky; it is also the result of a person observing, interpreting, remembering, and submitting a report through a particular reporting system.

**Research question:** Are UFO reports randomly scattered, or are they structured by time, place, shape language, duration, and reporting infrastructure?

**Dataset choice:** We chose the NUFORC-style UFO reporting archive because it is large, text-rich, and (crucially) shaped by *reporting infrastructure* (language, geography, and the rise of online submission). That makes it well suited for a story about how uncertainty becomes data.

**Audience:** curious non-technical readers who should be able to understand the findings from the website alone.

**Goal:** use narrative visualization to show that the dataset is patterned, biased, and deeply human without claiming that the reports verify UFO events. The updated story focuses especially on why the United States dominates the archive and what changes before and after internet-era reporting.


## 2. Basic Stats

Maven Analytics describes the UFO Sightings dataset as 80,000+ records between 1949 and 2014, with city, state, country, coordinates, shape, duration, date/time, and comments. The raw file used in this project is the archived NUFORC scrubbed CSV from Zenodo, which follows the same data lineage.

For the website, we ship a set of compact JSON aggregates in `site/data/` (and mirrored in `data/`) that drive all plots. The notebook reads those same JSON files so the explainer stays lightweight and consistent with what the public website shows.

In [ ]:
from pathlib import Path
import json

ROOT = Path('..').resolve()
SUMMARY = ROOT / 'site' / 'data' / 'summary.json'

summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary


### Cleaning Choices

The website is driven by precomputed JSON aggregates in `site/data/`. Those aggregates come from a cleaning + preprocessing pipeline applied to the archived NUFORC scrubbed data. Key choices:

- Parse the raw `datetime` field, including the NUFORC convention where some rows use hour `24`, which is rolled to `00:00` on the following day.
- Extract `year`, `month`, `weekday`, and `hour` for temporal analysis.
- Standardize country codes, state codes, and shape labels.
- Convert duration seconds to numeric values, marking invalid or negative durations as missing.
- Keep comments as text after HTML entity cleanup.
- Exclude invalid coordinates from map aggregates, but document how many were removed.

(We keep the explainer notebook focused on the *analysis outputs* that power the site: the JSON exports, not the full raw data file.)

In [ ]:
print(f"Raw rows: {summary['raw_rows']:,}")
print(f"Clean rows: {summary['clean_rows']:,}")
print(f"Valid coordinate rows: {summary['valid_geo_rows']:,}")
print(f"Actual date range: {summary['date_range']['min_year']}-{summary['date_range']['max_year']}")
print(f"Maven stated range: {summary['maven_stated_range']}")
print(f"Documented cleaning issues: {summary['issues']}")

The actual raw file contains reports from 1906 to 2014, while Maven states 1949 to 2014. This mismatch is treated as a provenance and data-quality issue, not hidden. It supports the broader point that the data is a historical reporting record rather than a perfectly curated event database.

## 3. Data Analysis

The analysis focuses on seven linked questions: when reports spike, what people say they saw, whether reports follow human rhythms, why the United States dominates, which US states and cities stand out, how duration changes by shape and era, and whether hotspots persist over time.

**Machine learning:** This project does not rely on a predictive model; the focus is explanatory and communicative rather than forecasting. Where we do compute derived measures (e.g., persistence ratio, burstiness), they are simple, interpretable summary statistics designed to support the narrative.

In [ ]:
def load_site_json(name):
    return json.loads((ROOT / 'site' / 'data' / name).read_text())

annual = load_site_json('annual_reports.json')
shapes = load_site_json('shape_counts.json')
countries = load_site_json('country_counts.json')
states = load_site_json('state_counts.json')
cities = load_site_json('city_counts.json')
duration_shapes = load_site_json('duration_by_shape.json')
duration_eras = load_site_json('duration_by_era_shape.json')
area51 = load_site_json('area51_summary.json')
hotspots = load_site_json('hotspots.json')

peak_year = max(annual, key=lambda d: d['reports'])
top_shapes = shapes[:8]
top_countries = countries[:5]
top_persistent = hotspots[:5]

print('Peak reporting year:', peak_year)
print('Top shapes:', top_shapes)
print('Top countries:', top_countries)
print('Top US cities:', cities[:8])
print('Duration by shape:', duration_shapes[:6])
print('Era comparison:', duration_eras)
print('Area 51 summary:', area51)
print('Most persistent map bins:', top_persistent)


### Exploratory Plots

This section is an inventory of **all plots on the website**, with notebook-friendly descriptions. Some figures are embedded here as static SVG summaries (stored in `notebooks/figures/`); other views are primarily interactive on the website (toggles, sliders, tooltips) and are described rather than duplicated.

**Website plot list (in page order):**

- **Hero world overview map** (`#hero-map`): introductory global context showing report-density clusters as proportional circles.
- **Dataset summary metrics** (`#summary-metrics`): clean row count, US share, raw date range, and the most common shape.
- **Top reported shapes** (`#shape-bars`): horizontal bar chart of the most common shape labels.
- **Shape groupings** (`#shape-group-stack` + legend): compresses many messy labels into a few visual-impression groups (lights, round forms, uncertain labels, structured forms).
- **Reports by year** (`#annual-line`): annual report volume with a linear/log scale toggle.
- **Annotated annual timeline** (`#annual-annotated`): a second annual view with cultural markers (e.g., *The X-Files*) and a highlighted internet-era split.
- **Month–hour heatmap** (`#heatmap`): reports by month of year and hour of day to show human observation rhythms.
- **US states (rank + choropleth)** (`#state-bars` + `#state-map`): raw vs per-capita (per million) toggles in both a ranking and a map.
- **US city hotspots** (`#city-bars`): top cities by report count.
- **Area 51 comparison** (`#area51-callout`): a small myth-vs-data sidebar comparing a bounding region near Area 51 with Las Vegas and Nevada totals.
- **Duration by shape (median)** (`#duration-bars`): median reported duration by common shape.
- **Duration by shape (distribution)** (`#duration-box`): log-scale distribution/quantiles to show how wide and skewed durations are.
- **Era comparison** (`#era-cards`): pre-1995 vs 1995–2014 changes in volume, top shape, and median duration.
- **Common words** (`#word-cloud`): most frequent terms in the comment text to show the ordinary observational vocabulary.
- **Hotspot exploration map** (`#hotspot-map`): decade slider + mode toggle (decade count vs persistence) with hover tooltips.
- **Country ranking** (`#country-bars`): a final global perspective (mostly a measure of reporting-pipeline reach, not “UFO activity”).

In [ ]:
# Rebuild the website aggregates (optional) and notebook SVG figures.
# The SVGs are written to both notebooks/figures/ and site/notebooks/figures/.
import subprocess
import sys

subprocess.run([sys.executable, str(ROOT / 'scripts' / 'build_data.py')], check=True)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'build_notebook_figures.py')], check=True)
print('Rebuilt site/data and regenerated notebook SVGs.')


#### Reports by year

The yearly trend shows why the project treats the data as a reporting system. Reports increase sharply in the 1990s and 2000s, which is consistent with easier online reporting and growing visibility of UFO reporting systems, but not proof that the sky itself changed.

The project uses 1995 as a practical internet-era split. Before 1995 there are far fewer reports and the median reported duration is longer; from 1995 onward, the archive becomes much denser and short observations appear more often.

![Reports by year](figures/annual_reports.svg)

A second view adds cultural markers and emphasizes interpretability rather than causality.

![Annual reports (annotated)](figures/annual_annotated.svg)


#### Top reported shapes

Shape labels reveal the vocabulary people use to classify uncertainty. `light` dominates overall, reinforcing that many reports describe distant or ambiguous visual phenomena. The shape vocabulary also changes over time: before 1995, `disk` is the leading shape; from 1995 to 2014, `light` becomes dominant.

The website also shows a grouped view that collapses many labels into a few broad visual-impression buckets (lights/flashes, round forms, uncertain labels, structured forms). That grouping is interpretive: it is meant to summarize *how people describe uncertainty*, not to claim the labels refer to the same physical object.

![Top reported shapes](figures/shape_counts.svg)

![Grouped shape impressions](figures/shape_groups.svg)


#### Month-hour rhythm

The heatmap connects reports to human routines: sightings cluster around evening and seasonal windows when people are more likely to be outside and looking upward.

![Reports by month and hour](figures/month_hour_heatmap.svg)

On the website, this view is also rendered as an interactive Plotly heatmap (`#heatmap`) with hover values.

#### Hotspot persistence

The persistence view separates high-volume bins from bins that keep appearing across many different years. This distinction addresses whether reports recur in the same places over time.

On the website, the exploratory map (`#hotspot-map`) adds a decade slider (to see changing hotspots over time), a mode toggle (decade counts vs persistence), and hover tooltips with bin-level summary stats.

![Hotspot persistence](figures/hotspot_persistence.svg)

A static snapshot of the decade-slider view (here: the 2000s) shows how the same bins can flare up in one period.

![Hotspot decade snapshot](figures/hotspot_decade_2000.svg)


#### USA state and city geography

The United States accounts for about 81% of clean reports, so the updated website zooms into US geography instead of leaving the map at a broad world overview. Raw state counts are led by large states such as California, Florida, Texas, and New York, but reports per million people bring Washington, Oregon, Montana, Alaska, Maine, Vermont, Arizona, New Mexico, and other smaller states forward.

On the website, this is shown as both a **ranking** and a **choropleth** (`#state-bars` + `#state-map`) with a raw/per-million toggle so the denominator effect is visible immediately. Below are static versions of the ranking view.

![State ranking (raw)](figures/state_bars_reports.svg)

![State ranking (per million)](figures/state_bars_per_million.svg)

City-level hotspots make the geography more concrete. Seattle, Phoenix, Las Vegas, Los Angeles, San Diego, Portland, Houston, and Chicago are among the largest city clusters (shown on the website as a city bar chart, `#city-bars`).

![US city hotspots](figures/city_hotspots.svg)


#### Area 51 as a reality check

Area 51 is useful as a cultural reference, but it is not the main count story in this dataset. Using a simple bounding region around the Area 51/Rachel/Groom Lake area, there are about 20 nearby reports. Nevada has 803 reports overall, and Las Vegas alone has 363. This makes Area 51 a good sidebar: famous in the imagination, small in the archive.

![Area 51 comparison](figures/area51_comparison.svg)

On the website, this appears as a small comparison bar chart (`#area51-callout`).


#### Duration by shape

Reported duration is noisy because it is estimated by people, but it still reveals a relationship between event description and event length. Among common shapes, `changing` reports have the longest median duration, followed by shapes such as `diamond` and `disk`. Short labels such as `flash` and `chevron` tend to have short medians.

A median-only view is simple, but it can hide how wide these distributions are.

![Duration by shape](figures/duration_by_shape.svg)

A log-scale distribution view makes the spread visible (box = p25–p75, whiskers = p05–p95).

![Duration distribution by shape](figures/duration_boxplot.svg)


#### Before and after internet-era reporting

Using 1995 as the split, the archive changes strongly. Pre-internet reports have a median duration around 300 seconds and `disk` is the leading shape. In the 1995-2014 internet era, median duration drops to about 180 seconds and `light` becomes the leading shape. This is interpreted as a change in reporting behavior and vocabulary, not proof that physical events changed.


#### Common words in comments

The website includes a word-focused view (`#word-cloud`) built from the most frequent tokens in the comment text. The key takeaway is that the vocabulary is largely observational rather than sensational: lights, brightness, motion, color, speed, and formations dominate. This supports the project’s framing that many entries are brief descriptions of ambiguous visual phenomena rather than detailed close-contact narratives.

![Common words](figures/word_cloud.svg)


#### Global perspective (country ranking)

The final website chart (`#country-bars`) ranks the top reporting countries to make the reporting-pipeline bias visible: the dataset is heavily US-centered, with a thin fringe of English-adjacent countries (Canada, UK, Australia, etc.). This is treated as evidence about *where the reporting system is reachable*, not as a balanced global measure of UFO activity.

![Country ranking](figures/country_bars.svg)


### Bias Analysis

| Bias | Why it matters | How the project handles it |
| --- | --- | --- |
| US reporting-system bias | About 81% of clean reports are coded as United States. | Treat US dominance as evidence of NUFORC visibility, English-language access, population, internet access, and reporting culture. |
| Population bias | More people means more possible reports. | Show raw geography carefully and include a simple US population-normalized state view. |
| Internet/reporting bias | Online reporting changed who could submit reports and when. | Compare pre-1995 with 1995-2014 and avoid causal certainty. |
| Language bias | English-speaking countries are overrepresented. | Discuss country concentration as reporting-system coverage, not true UFO activity. |
| Geocoding bias | Coordinates may point to city centers rather than exact locations. | Use spatial bins and city/state aggregates instead of exact point claims. |
| Selection bias | Only people who report are included. | Describe records as reports, never verified sightings. |
| Visibility bias | Darkness, weather, season, and outdoor activity affect observation. | Use month-hour patterns as behavioral context. |


### Hotspot Persistence

A simple count can confuse a one-time burst with a place that keeps appearing. The website therefore aggregates reports into coarse spatial bins and computes a small set of interpretable per-bin summary statistics.

For each bin (see `site/data/hotspots.json`):

- `total_reports`: total reports in the bin across the full archive
- `active_years`: how many distinct years have at least one report in the bin
- `active_decades`: how many distinct decades have at least one report in the bin
- `persistence_ratio`: a normalized “keeps showing up” measure (
  \(\text{active\_years} / \text{observed\_year\_span}\)
  )
- `burstiness`: the fraction of the bin’s reports that occur in its busiest year (high values indicate “spike-y” hotspots)

The interactive map combines two related datasets:

- `site/data/hex_decade_bins.json`: per-decade counts for the slider view
- `site/data/hotspots.json`: cross-archive persistence stats for the persistence view

This is enough to address the “do hotspots persist?” question without overclaiming a predictive model.

## 4. Genre

**Chosen genre:** a *martini glass* narrative (Segel & Heer). Most of the page is author-driven, with a linear path through the argument. Near the end, the story “opens up” into a more reader-driven exploratory view (the hotspot map with a decade slider and mode toggle).

### Figure 7 taxonomy: Visual Narrative

Segel & Heer group visual narrative techniques into three categories. Here is how the website uses each category.

#### 1) Visual structuring

- The page is organized as a sequence of **chapters** with one main question each (e.g., shapes → time → rhythms → US geography → duration → hotspots).
- The same chart styling and figure-note pattern is repeated throughout to create visual continuity.
- The top navigation links (e.g., `#story`, `#usa`, `#duration`, `#hotspots`) make the narrative structure legible.

#### 2) Highlighting

- The annual report chart highlights the internet-era split (1995) as a visual annotation cue (`#annual-line`).
- The annotated annual timeline adds explicit markers (e.g., *The X-Files* premiere) to focus attention on interpretable inflection points (`#annual-annotated`).
- “Sidebar” comparisons (e.g., Area 51 vs Las Vegas/Nevada) are used as a deliberate expectation-check highlight (`#area51-callout`).

#### 3) Transition guidance

- The narrative flows from global context (hero overview) to increasingly specific evidence (US states → cities) and finally to exploration (hotspot map).
- Small toggles create guided transitions without asking the reader to re-learn the visualization:
  - linear/log scale on the annual chart (`#annual-line`)
  - raw vs per-million in the state view (`#state-bars` + `#state-map`)
  - decade-count vs persistence in the hotspot view (`#hotspot-map`)

### Figure 7 taxonomy: Narrative Structure

Segel & Heer group narrative structure techniques into three categories. Here is how the website uses each category.

#### 1) Ordering

- The narrative is primarily **linear** and scroll-based: each section sets up the next, so the reader is rarely asked to interpret a plot without context.
- The ordering is designed to prevent a common misread: raw geography → *then* denominator corrections (per-million) → *then* local hotspots.

#### 2) Interactivity

Interactivity is used for *inspection and comparison*, not freeform sandboxing:

- Toggle interactions for alternative encodings of the same story question (scale, normalization, map mode).
- Hover tooltips on Plotly charts provide precise values without clutter.
- The hotspot map is the most reader-driven element: decade slider, mode toggle, and hover details (`#hotspot-map`).

#### 3) Messaging

- Each chart is paired with a **non-technical takeaway** in the surrounding text and a figure note (what to notice, why it matters, and what it does *not* prove).
- Repeated caveats (reports ≠ verified events; reporting infrastructure matters; denominators matter) act as the “guard rails” of the story.


## 5. Visualizations

| Visualization | Purpose | Interaction | Limitation |
| --- | --- | --- | --- |
| Hero map | Create the initial sense of geographic pattern. | Static overview. | Raw geography can be mistaken for true activity. |
| Dataset metrics | Ground the reader in row count, date range, US share, and top shape. | Static. | Summary stats cannot show every cleaning choice. |
| Shape bar chart | Show the vocabulary of reported sightings. | Static. | Shape labels are self-reported and messy. |
| Shape grouping stack | Summarize many labels into a few visual-impression groups. | Static. | Grouping is interpretive and lossy. |
| Annual line chart | Show temporal growth and reporting spikes. | Linear/log toggle. | Cannot prove the internet caused the rise. |
| Annotated annual timeline | Add cultural markers and highlight key inflections. | Static annotations. | Markers are contextual, not causal. |
| Month-hour heatmap | Show human observation rhythms. | Hover values (website). | Does not include weather or true sky-watching rates. |
| State ranking + choropleth | Compare raw counts with reports per million. | Toggle raw/per-million. | 2010 population is an imperfect historical denominator. |
| City hotspots | Move from states toward recognizable places. | Static ranking. | City counts inherit geocoding and population bias. |
| Area 51 comparison | Myth-vs-data sidebar. | Static ranking. | Bounding region is a rough proxy for “Area 51.” |
| Duration by shape (median) | Compare median duration across shape labels. | Static ranking. | Durations are human estimates. |
| Duration by shape (distribution) | Show quantiles on a log scale to reveal skew. | Hover values (website). | Extreme values and logging can be misread. |
| Era comparison | Compare pre-1995 reports with 1995-2014 reports. | Hover values (website). | The split is interpretive, not causal proof. |
| Word cloud | Summarize common comment vocabulary. | Hover values (website). | Tokenization choices affect which words dominate. |
| Hotspot exploration map | Distinguish decade bursts from persistent places. | Time slider, count/persistence toggle, hover tooltips. | Approximate hex bins are not equal-area H3 cells. |
| Country ranking | Provide a final global perspective. | Static ranking. | Country coverage reflects the reporting pipeline. |


## 6. Discussion

What went well: the dataset supports a strong social-data story because temporal, geographic, linguistic, duration, and reporting-system patterns all point in the same direction. The United States focus makes the geographic argument clearer: raw counts identify populous and reporting-heavy states, normalized counts reveal smaller high-rate states, and city hotspots show recognizable urban reporting clusters.

Main findings: about 81% of clean reports are coded as US reports; California leads raw state counts, while Washington and several smaller western/northern states stand out per million people. Seattle, Phoenix, Las Vegas, Los Angeles, San Diego, Portland, Houston, and Chicago are among the largest city hotspots. Area 51 is culturally famous, but the nearby bounding region has only about 20 reports in this data, compared with 363 reports from Las Vegas. Duration also changes: among common shapes, `changing` has the longest median duration, while `flash` and `chevron` are short; pre-1995 reports have a median duration of about 300 seconds, while the 1995-2014 internet era has a median around 180 seconds.

What is missing: stronger causal claims would require external datasets such as historical internet adoption, media coverage, local population by year, weather, light pollution, or aircraft/airport activity. The current population normalization is deliberately simple and should be presented as a caveat rather than a final correction.

Main limitation: the records are reports, not verified UFO events. The website therefore avoids saying where UFOs actually are and instead asks what the reports reveal about people, places, and reporting behavior. The playful closing joke is aimed at American reporting culture: Americans may not be uniquely good at spotting aliens, but they are very good at seeing something weird and finding a form to submit.


## 7. Contributions

- Marcus Elkjær: ________________________________
- Martin Jespersen: ______________________________
- Sebastian Wulf-Andersen: ________________________


## 8. References

- Segel, E., & Heer, J. (2010). *Narrative Visualization: Telling Stories with Data*. IEEE Transactions on Visualization and Computer Graphics. `http://vis.stanford.edu/files/2010-Narrative-InfoVis.pdf`
- Maven Analytics. (n.d.). *UFO Sightings dataset page*. `https://mavenanalytics.io/data-playground/ufo-sightings`
- NUFORC scrubbed archive (Zenodo). (n.d.). *Archived NUFORC scrubbed CSV record*. Zenodo. `https://zenodo.org/records/1205624`
- National UFO Reporting Center (NUFORC). (n.d.). *About NUFORC*. `https://nuforc.org/about-us/`
- Pew Research Center. (n.d.). *Internet/Broadband fact sheet* (used for contextual discussion of reporting infrastructure). `https://www.pewresearch.org/internet/fact-sheet/internet-broadband/`
- Course requirements. (2026). *Social Data Analysis and Visualization: Final Project*. `https://github.com/suneman/socialdata2026/wiki/Final-Project`

(Where external context is used in the website text, it is treated as supporting evidence for interpretation, not as ground-truth verification of any individual report.)